# ADNI WFCM 

PCA uses 12 explicitly listed clinical predictors from the 2,436-subject cohort. The first **three** components are retained for the three-dimensional analysis and explain **70.3383%** of total variance. 

We fix k=2 for an AD-like versus non-AD-like geometric spectrum and select m by minimum refined numerical NLL. The choice follows exploratory analysis and is not claimed to have been originally prespecified or data-selected as optimal. All fitting, bootstrap inference and figures use the three-PC representation generated by the preprocessing below.

### Required data and preparation
The raw ADNI package (`ADNIMERGE_0.0.1.tar.gz`) and all individual-level ADNI CSV files are **not included in the repository**, due to data-use restrictions. Obtain authorized ADNI/IDA access, then obtain the ADNIMERGE package using the repository's access instructions. This notebook does not download restricted records.

Default input: local `data/ADNI_data.csv`, the baseline-derived study table with the 12 named predictors below and diagnosis field `DX`. This table must have the same cohort and row order as the study input. Exporting the underlying ADNI table from the approved source is a separate data preparation requirement; the notebook does not claim to reconstruct the cohort from an arbitrary release of ADNIMERGE. The reference has 2,436 rows and 2,422 recorded diagnoses. The embedded preprocessing performs columnwise median imputation, standardizes each predictor using sample SD (ddof=1), computes full-SVD PCA and fixes component signs by the largest absolute loading (independently of labels). It exports `ADNI_pc_final.csv`, loadings, eigenvalues, preprocessing statistics and provenance under `results_adni_final/`. It then re-standardizes PC1–PC3 using population SD before WFCM fitting.

AD/Dementia diagnoses annotate AD; CN/MCI/EMCI/LMCI/SMC annotate non-AD. Missing diagnoses are excluded from label metrics, not unsupervised fitting. Labels do not select m or initialize centers. Memberships are geometric fuzzy allocations, not calibrated AD probabilities or longitudinal progression estimates.


## Setup
This is **one self-contained code file**: all WFCM fitting, integration, inference and plotting functions are embedded. **Datasets are separate inputs**, not embedded in this notebook. Start a fresh Python kernel, configure the data path, then run every cell in order. 

Tested Python: 3.12.9. A matching environment can be installed with:
```python
%pip install numpy==2.5.4 scipy==1.18.1 pandas==3.1.0 scikit-learn==1.9.1 matplotlib==3.11.2
```
Run the installation command separately if needed; this notebook does not automatically modify your environment. Full inference uses B=100 and can be slow. Decreasing B or the integration budget is a diagnostic run, not reproduction of the reference result.


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ.setdefault('OMP_MAX_ACTIVE_LEVELS','1')
os.environ.setdefault('LOKY_MAX_CPU_COUNT','1')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from scipy.stats import chi2
from pathlib import Path
# Portable data lookup; set WFCM_DATA_DIR or edit the input setting below.
import os, hashlib, platform
from pathlib import Path

def locate_data(filename, required=True):
    preferred=Path(os.environ.get('WFCM_DATA_DIR','data'))/filename
    candidates=[preferred,Path.cwd()/'data'/filename,Path.cwd().parent/'data'/filename,Path.cwd()/filename]
    for candidate in candidates:
        if candidate.is_file():return candidate.resolve()
    if required:
        raise FileNotFoundError(f'Required dataset {filename!r} is not bundled. Put it in data/, set WFCM_DATA_DIR, or edit the input path. No substitute data will be used.')
    return preferred.resolve()

# Save every displayed figure
import matplotlib.pyplot as plt
_base_show=getattr(plt.show,'_wfcm_base_show',plt.show)
_saved_figure_files=[]
def _show_and_save(*args,**kwargs):
    directory=Path(OUTPUT_DIR)/'figures';directory.mkdir(parents=True,exist_ok=True)
    for number in plt.get_fignums():
        figure=plt.figure(number)
        if not getattr(figure,'_wfcm_exported',False):
            file=directory/f'figure_{len(_saved_figure_files)+1:02d}.png'
            figure.savefig(file,dpi=180,bbox_inches='tight');figure._wfcm_exported=True
            _saved_figure_files.append(str(file))
    return _base_show(*args,**kwargs)
_show_and_save._wfcm_base_show=_base_show
plt.show=_show_and_save


In [ ]:
"""ADNI Figure 8: Algorithm 2 / Eq. (13) """
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def ellipsoid_surface(mean,covariance,cutoff,n_azimuth=64,n_polar=32):
    """(v-mean)' covariance^{-1} (v-mean)=cutoff; no extra B or n scaling."""
    mean=np.asarray(mean,dtype=float);cov=np.asarray(covariance,dtype=float)
    if mean.shape!=(3,) or cov.shape!=(3,3) or not np.isfinite(mean).all() or not np.isfinite(cov).all() or not np.allclose(cov,cov.T,rtol=1e-10,atol=1e-12) or not np.isfinite(cutoff) or cutoff<=0:
        raise ValueError('Require a finite 3D mean, symmetric covariance and positive cutoff.')
    eigenvalues,eigenvectors=np.linalg.eigh(cov)
    if np.any(eigenvalues<=0):raise ValueError('Center covariance must be positive definite; no pseudoinverse or jitter used.')
    azimuth=np.linspace(0,2*np.pi,n_azimuth);polar=np.linspace(0,np.pi,n_polar)
    sphere=np.stack([np.outer(np.cos(azimuth),np.sin(polar)),np.outer(np.sin(azimuth),np.sin(polar)),np.outer(np.ones_like(azimuth),np.cos(polar))],axis=-1)
    radii=np.sqrt(cutoff*eigenvalues)
    return sphere@(eigenvectors*radii).T+mean

def draw_center_regions(ax,X,inference,ad_index):
    means=np.asarray(inference['center_means']);cov=np.asarray(inference['center_covariance']);cutoff=float(inference['center_cutoff'])
    if means.shape!=(2,3) or cov.shape!=(2,3,3) or ad_index not in (0,1):raise ValueError('ADNI plot requires two aligned 3D centers and an AD-like annotation.')
    X=np.asarray(X);ax.scatter(*X.T,s=7,alpha=.30,c='steelblue',edgecolors='none',depthshade=False,label='Data')
    vertices=[]
    for j in range(2):
        color='#b40426' if j==ad_index else '#3b4cc0';name='AD-like' if j==ad_index else 'Non-AD-like'
        surface=ellipsoid_surface(means[j],cov[j],cutoff);vertices.append(surface.reshape(-1,3))
        ax.plot_surface(surface[:,:,0],surface[:,:,1],surface[:,:,2],color=color,alpha=.25,linewidth=0,antialiased=True)
        ax.plot_wireframe(surface[:,:,0],surface[:,:,1],surface[:,:,2],color=color,alpha=.55,rstride=8,cstride=4,linewidth=.4)
        ax.scatter(*means[j],marker='X',s=36,color=color,edgecolors='black',linewidths=.35,label=f'{name} mean',depthshade=False)
    # Include complete ellipsoids rather than clipping uncertainty to the data range.
    limits=np.vstack([X,*vertices])
    for column,setlim in enumerate([ax.set_xlim,ax.set_ylim,ax.set_zlim]):
        lower,upper=limits[:,column].min(),limits[:,column].max();pad=max(.05*(upper-lower),.1);setlim(lower-pad,upper+pad)
    ax.set(xlabel='PC1',ylabel='PC2',zlabel='PC3');ax.view_init(elev=22,azim=-60);ax.legend(loc='upper right',fontsize=9)
    return ax

def draw_membership_line(ax,U,ad_index):
    U=np.asarray(U,dtype=float)
    if U.ndim!=2 or U.shape[1]!=2 or np.any(U<0) or not np.allclose(U.sum(axis=1),1.):raise ValueError('Require actual two-cluster simplex memberships.')
    nonad=1-ad_index;x=U[:,nonad];colors=np.zeros((len(U),3));colors[:,0]=U[:,ad_index];colors[:,2]=U[:,nonad]
    ax.plot([0,1],[0,0],color='gray',alpha=.45,zorder=0)
    ax.scatter(x,np.zeros(len(x)),c=colors,s=15,alpha=.45,edgecolors='none')
    ax.scatter([0,1],[0,0],c=['#b40426','#3b4cc0'],s=90,zorder=3)
    ax.set(xlim=(-.06,1.06),ylim=(-.08,.08),xlabel='Membership to non-AD-like prototype',yticks=[])
    ax.set_xticks([0,.25,.5,.75,1],['0\nAD-like','0.25','0.50','0.75','1\nNon-AD-like'])
    ax.set_title('Soft Cluster Memberships on 2D Simplex (AD vs non-AD)',fontsize=11)
    return ax

def save_adni_confidence_figures(X,U,inference,ad_index,output_dir):
    """Save a standalone center plot and the paired manuscript Figure 8 layout."""
    output=Path(output_dir);output.mkdir(parents=True,exist_ok=True)
    level=100*(1-float(inference['alpha']))
    fig=plt.figure(figsize=(10,8));ax=fig.add_subplot(111,projection='3d');draw_center_regions(ax,X,inference,ad_index)
    ax.set_title(f'3D {level:g}% Bootstrap Confidence Ellipsoids for 2 Centers',pad=18)
    fig.subplots_adjust(left=.03,right=.96,bottom=.05,top=.92)
    for suffix in ['png','svg']:fig.savefig(output/f'adni_ci.{suffix}',dpi=220,bbox_inches='tight',facecolor='white')
    paired=plt.figure(figsize=(15,6.5));left=paired.add_subplot(121,projection='3d');right=paired.add_subplot(122)
    draw_center_regions(left,X,inference,ad_index);left.set_title(f'{level:g}% Center Confidence Ellipsoids',pad=15)
    draw_membership_line(right,U,ad_index);paired.subplots_adjust(left=.02,right=.98,bottom=.16,top=.90,wspace=.12)
    for suffix in ['png','svg']:paired.savefig(output/f'adni_figure8_centers_simplex.{suffix}',dpi=220,bbox_inches='tight',facecolor='white')
    # Supplemental local views make small, correctly scaled regions visible.
    zoom=plt.figure(figsize=(12,5.5))
    for j in range(2):
        detail=zoom.add_subplot(1,2,j+1,projection='3d');color='#b40426' if j==ad_index else '#3b4cc0'
        mean=np.asarray(inference['center_means'][j]);surface=ellipsoid_surface(mean,inference['center_covariance'][j],inference['center_cutoff'])
        detail.plot_surface(surface[:,:,0],surface[:,:,1],surface[:,:,2],color=color,alpha=.28,linewidth=0)
        detail.plot_wireframe(surface[:,:,0],surface[:,:,1],surface[:,:,2],color=color,alpha=.6,rstride=8,cstride=4,linewidth=.4)
        detail.scatter(*mean,color=color,marker='X',s=35,edgecolors='black',linewidths=.3)
        vertices=surface.reshape(-1,3)
        for axis,setlim in enumerate([detail.set_xlim,detail.set_ylim,detail.set_zlim]):
            low,high=vertices[:,axis].min(),vertices[:,axis].max();pad=max(.08*(high-low),1e-5);setlim(low-pad,high+pad)
        detail.set(xlabel='PC1',ylabel='PC2',zlabel='PC3',title=('AD-like' if j==ad_index else 'Non-AD-like')+' center: local view');detail.view_init(elev=22,azim=-60)
    zoom.suptitle(f'{level:g}% confidence-region details');zoom.subplots_adjust(left=.02,right=.96,top=.88,bottom=.08,wspace=.18)
    for suffix in ['png','svg']:zoom.savefig(output/f'adni_ci_zoom.{suffix}',dpi=220,bbox_inches='tight',facecolor='white')
    np.savez_compressed(output/'adni_ellipsoid_geometry.npz',center_means=inference['center_means'],center_covariance=inference['center_covariance'],center_precision=inference['center_precision'],chi2_cutoff=inference['center_cutoff'],alpha=inference['alpha'],ad_index=ad_index)
    return fig,paired

def write_adni_table2(inference,output_dir):
    """Export aligned bootstrap means, denominator-B SDs and Algorithm-2 intervals."""
    import pandas as pd
    output=Path(output_dir);output.mkdir(parents=True,exist_ok=True)
    if inference['k']!=2 or inference['d']!=3:raise ValueError('ADNI Table 2 requires k=2,d=3.')
    names=['sigma']+[f'v{j+1}_{axis}' for j in range(2) for axis in ['PC1','PC2','PC3']]+['w1','w2']
    table=pd.DataFrame(dict(parameter=names,bootstrap_mean=inference['mean'],bootstrap_sd_ddof0=inference['sd'],lower=inference['ci'][:,0],upper=inference['ci'][:,1]))
    table.to_csv(output/'adni_table2_bootstrap.csv',index=False)
    texnames=[r'\sigma']+[rf'v_{{{j+1}}}^{{{axis}}}' for j in range(2) for axis in ['x','y','z']]+['w_1','w_2']
    lines=[r'\begin{table}[htbp]',r'\centering',rf'\caption{{ADNI parametric bootstrap summary statistics and {100*(1-inference["alpha"]):g}\% confidence intervals.}}',r'\begin{tabular}{lcc}',r'\hline',rf'Parameter & Mean $\pm$ Std. & {100*(1-inference["alpha"]):g}\% CI \\',r'\hline']
    for name,mean,sd,ci in zip(texnames,inference['mean'],inference['sd'],inference['ci']):
        lines.append(rf'${name}$ & ${mean:.3f}\pm {sd:.3f}$ & $[{ci[0]:.3f}, {ci[1]:.3f}]$ \\')
    lines += [r'\hline',r'\end{tabular}',r'\end{table}']
    (output/'adni_table2_bootstrap.tex').write_text('\n'.join(lines)+'\n')
    return table


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Ellipse

def pc_axes(fig,position,d):
    if d not in (2,3):raise ValueError('Plotting supports two or three analysis PCs.')
    return fig.add_subplot(position,**({'projection':'3d'} if d==3 else {}))
def label_pc_axes(ax,d,title):
    ax.set(xlabel='PC1',ylabel='PC2',title=title)
    if d==3:ax.set_zlabel('PC3',labelpad=0)
def plot_pc_fit(X,centers,k,m):
    d=X.shape[1];fig=plt.figure(figsize=(8,6));ax=pc_axes(fig,111,d);ax.scatter(*X.T,s=8,alpha=.25);ax.scatter(*centers.T,c='black',marker='X',s=100);label_pc_axes(ax,d,f'Fixed k={k}; fitted m={m:.4g}');fig.tight_layout();return fig

def compare_pc_memberships(X,panels,lower=.05,upper=.09,
                           *, show_left_colorbar=False):                   
    d=X.shape[1];fig=plt.figure(figsize=(14,6))
    for position,(values,centers,title,rescaled,binary) in enumerate(panels,1):
        ax=pc_axes(fig,120+position,d);values=np.asarray(values,dtype=float);norm=Normalize(lower,upper,clip=True) if rescaled else Normalize(0,1)
        if binary:
            colors=plt.cm.coolwarm(norm(np.nan_to_num(values,nan=.5))); colors[~np.isfinite(values)]=[.6,.6,.6,1]
            ax.scatter(*X.T,c=colors,s=15,alpha=.85);mappable=ScalarMappable(norm=norm,cmap='coolwarm')
        else:mappable=ax.scatter(*X.T,c=values,norm=norm,cmap='coolwarm',s=15,alpha=.85)
        ax.scatter(*centers.T,c='black',marker='X',s=100);label_pc_axes(ax,d,title)
        if position != 1 or show_left_colorbar:
            fig.colorbar(mappable,ax=ax,shrink=.65,pad=.14,fraction=.04,extend='both' if rescaled else 'neither',label='Raw membership (clipped display)' if rescaled else 'AD Membership' if binary else 'Membership')
    fig.tight_layout();return fig

def _draw_2d_regions(ax,X,inference,ad_index):
    ax.scatter(*X.T,s=8,c='steelblue',alpha=.25)
    for j in range(2):
        C=inference['center_covariance'][j];e,Q=np.linalg.eigh(C)
        if np.any(e<=0):raise ValueError('Positive-definite center covariance required.')
        angle=np.degrees(np.arctan2(Q[1,0],Q[0,0]));r=np.sqrt(e*inference['center_cutoff']);color='firebrick' if j==ad_index else 'royalblue';mean=inference['center_means'][j]
        ax.add_patch(Ellipse(mean,2*r[0],2*r[1],angle=angle,facecolor=color,edgecolor=color,alpha=.3));ax.scatter(*mean,c=color,marker='X',s=70,label=('AD-like' if j==ad_index else 'Non-AD-like')+' mean')
    ax.autoscale_view();label_pc_axes(ax,2,'95% bootstrap center confidence ellipses');ax.legend(fontsize=9)

def save_adni_regions(X,U,inference,ad_index,output_dir):
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    if X.shape[1]!=inference['d']:raise ValueError('Confidence-region dimension must equal fitted feature dimension.')
    if inference['d']==3:return save_adni_confidence_figures(X,U,inference,ad_index,out)
    if inference['d']!=2:raise ValueError('Only 2D and 3D plots supported.')
    fig,ax=plt.subplots(figsize=(8,7));_draw_2d_regions(ax,X,inference,ad_index);fig.tight_layout()
    pair,axes=plt.subplots(1,2,figsize=(14,6));_draw_2d_regions(axes[0],X,inference,ad_index);draw_membership_line(axes[1],U,ad_index);pair.tight_layout()
    for suffix in ['png','svg']:
        fig.savefig(out/f'adni_ci.{suffix}',dpi=220,bbox_inches='tight');pair.savefig(out/f'adni_centers_simplex.{suffix}',dpi=220,bbox_inches='tight')
    return fig,pair

def write_adni_parameter_table(inference,output_dir):
    d=inference['d'];names=['sigma']+[f'v{a+1}_PC{j+1}' for a in range(2) for j in range(d)]+['w1','w2']
    table=pd.DataFrame(dict(parameter=names,bootstrap_mean=inference['mean'],bootstrap_sd_ddof0=inference['sd'],lower=inference['ci'][:,0],upper=inference['ci'][:,1]));out=Path(output_dir);out.mkdir(parents=True,exist_ok=True);table.to_csv(out/'adni_table2_bootstrap.csv',index=False);return table


In [ ]:
# Embedded preprocessing generates the final PCA-score dataset.
OUTPUT_DIR=Path('results_adni_final').resolve()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
INPUT_MODE='raw'  # 'raw' (rebuild PCA) or 'corrected_scores' (verified reference scores)
FEATURES=['AGE','PTEDUCAT','APOE4','CDRSB','ADAS11','ADAS13','ADASQ4','MMSE',
          'RAVLT.perc.forgetting','LDELTOTAL','TRABSCOR','FAQ']
PC_COLUMNS=['PC1','PC2','PC3']
EXPECTED_CORRECTED_SCORE_HASH='c12fe7431707e1b10fb568266f6a6c9e99c2760fedc15c353dd908247ce85209'

if INPUT_MODE=='raw':
    DATA_PATH=locate_data('ADNI_data.csv')
    source_df=pd.read_csv(DATA_PATH)
    required=FEATURES+['DX']
    if not set(required).issubset(source_df.columns):
        raise ValueError('ADNI_data.csv must contain the 12 listed clinical features and DX.')
    numeric=source_df[FEATURES].apply(pd.to_numeric,errors='raise')
    medians=numeric.median();imputed=numeric.fillna(medians)
    mean=imputed.mean().to_numpy();sample_sd=imputed.std(ddof=1).to_numpy()
    if not np.isfinite(imputed.to_numpy()).all() or np.any(sample_sd<=0):
        raise ValueError('PCA predictors must be finite after median imputation and have positive SD.')
    standardized=(imputed.to_numpy()-mean)/sample_sd
    _,singular,axes=np.linalg.svd(standardized,full_matrices=False)
    signs=np.sign(axes[np.arange(len(axes)),np.abs(axes).argmax(axis=1)])
    axes*=signs[:,None]
    scores=standardized@axes.T
    eigenvalues=singular**2/(len(standardized)-1)
    ratios=eigenvalues/eigenvalues.sum()
    component_names=[f'PC{i+1}' for i in range(len(eigenvalues))]
    df=pd.DataFrame(scores,columns=component_names)
    df['cluster']=source_df['DX'].to_numpy()
    pca_variance=pd.DataFrame(dict(component=component_names,eigenvalue=eigenvalues,
        standard_deviation=np.sqrt(eigenvalues),explained_variance_ratio=ratios,
        cumulative_explained_variance=ratios.cumsum(),retained_kaiser=eigenvalues>1,
        used_in_analysis=np.arange(len(eigenvalues))<3))
    df.to_csv(OUTPUT_DIR/'ADNI_pc_final.csv',index=False)
    pca_variance.to_csv(OUTPUT_DIR/'pca_variance.csv',index=False)
    pd.DataFrame(axes.T,index=FEATURES,columns=component_names).rename_axis('feature').to_csv(OUTPUT_DIR/'pca_loadings.csv')
    pd.DataFrame(dict(feature=FEATURES,median=medians.to_numpy(),mean=mean,sample_sd=sample_sd,
        imputed_count=numeric.isna().sum().to_numpy())).to_csv(OUTPUT_DIR/'pca_scaling.csv',index=False)
    np.savez_compressed(OUTPUT_DIR/'pca_model.npz',features=np.array(FEATURES),median=medians.to_numpy(),
        mean=mean,sample_sd=sample_sd,components=axes,eigenvalues=eigenvalues)
    pca_provenance=dict(features=FEATURES,diagnosis_columns=['DX','DX.bl'],
        n=len(df),input_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
        dimension_rule='first three PCs deliberately retained for 3D analysis; not Kaiser',
        analysis_pc_columns=PC_COLUMNS,first_three_explained_variance=float(ratios[:3].sum()),
        kaiser_retained=[name for name,keep in zip(component_names,eigenvalues>1) if keep],
        sign_convention='largest-absolute loading positive; independent of labels',
        standardization='sample SD ddof=1 before PCA; population SD before WFCM')
elif INPUT_MODE=='corrected_scores':
    DATA_PATH=locate_data('ADNI_pc_final.csv')
    df=pd.read_csv(DATA_PATH)
    if not set(PC_COLUMNS+['cluster']).issubset(df.columns):
        raise ValueError('Corrected score input must contain PC1, PC2, PC3 and cluster.')
    score_hash=hashlib.sha256(np.asarray(df[PC_COLUMNS],dtype='<f8').tobytes()).hexdigest()
    if score_hash!=EXPECTED_CORRECTED_SCORE_HASH:
        raise ValueError('Scores do not match the reference PCA scores. Use raw mode to rebuild PCA for a different input cohort.')
    # All eigenvalues below belong to the verified corrected reference preprocessing.
    eigenvalues=np.array([6.343183467597747, 1.1224618583869568, 0.9749524528516867, 0.7991077899660375, 0.7195693742716548, 0.5705721297754732, 0.4414438118568011, 0.3471229958545257, 0.31955132050879115, 0.19817881460598516, 0.15059885275380117, 0.01325713157054813])
    ratios=eigenvalues/eigenvalues.sum()
    component_names=[f'PC{i+1}' for i in range(len(eigenvalues))]
    pca_variance=pd.DataFrame(dict(component=component_names,eigenvalue=eigenvalues,
        standard_deviation=np.sqrt(eigenvalues),explained_variance_ratio=ratios,
        cumulative_explained_variance=ratios.cumsum(),retained_kaiser=eigenvalues>1,
        used_in_analysis=np.arange(len(eigenvalues))<3))
    pca_variance.to_csv(OUTPUT_DIR/'pca_variance.csv',index=False)
    pca_provenance=dict(features=FEATURES,diagnosis_columns=['DX','DX.bl'],
        n=len(df),input_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
        corrected_reference_scores_verified=True,analysis_pc_columns=PC_COLUMNS,
        dimension_rule='first three PCs deliberately retained for 3D analysis; not Kaiser',
        first_three_explained_variance=float(ratios[:3].sum()))
else:
    raise ValueError("INPUT_MODE must be 'raw' or 'corrected_scores'.")
if len(df)<4 or not np.isfinite(df[PC_COLUMNS].to_numpy(dtype=float)).all():
    raise ValueError('At least four finite corrected three-PC observations are required.')
(OUTPUT_DIR/'pca_provenance.json').write_text(__import__('json').dumps(pca_provenance,indent=2))
print('PCA clinical predictors:',FEATURES)
print('Analysis PCs:',PC_COLUMNS,'Explained variance:',pca_provenance['first_three_explained_variance'])


In [ ]:
scaler=StandardScaler()
df_scaled=pd.DataFrame(scaler.fit_transform(df[PC_COLUMNS]),columns=PC_COLUMNS,index=df.index)
D=len(PC_COLUMNS)


## PCA
The scree table reports all eigenvalues and identifies the first three analysis components. The three-PC representation explains 70.3383% of total variance. 

In [ ]:
# display(pca_variance)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(np.arange(1,len(pca_variance)+1),pca_variance.eigenvalue,'o-')
ax.axhline(1,color='gray',linestyle='--',label='Kaiser threshold (diagnostic)')
ax.axvspan(.5,3.5,color='steelblue',alpha=.12,label='Three analysis PCs')
ax.set(xlabel='Principal component',ylabel='Eigenvalue',title='PCA of 12 clinical predictors')
ax.legend();fig.tight_layout();plt.show()


In [ ]:
"""WFCM fitting and fixed-(k,m) parametric bootstrap from the manuscript.

Dependencies: numpy, scipy, scikit-learn. No data preprocessing is performed.
All likelihoods use f(x) proportional to exp(-h(x)/sigma**2), where
h(x) = [sum_j (w_j ||x-v_j||**2)**(-1/(m-1))]**(-(m-1)).
"""
import numpy as np
from scipy.optimize import minimize, linear_sum_assignment
from scipy.special import logsumexp
from scipy.spatial.distance import cdist
from scipy.stats import chi2, norm
from sklearn.mixture import GaussianMixture


def _data(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or min(X.shape) < 1 or not np.isfinite(X).all():
        raise ValueError('Expected a finite, nonempty samples-by-features matrix.')
    return X


from scipy.stats import multivariate_t

class DefensiveProposal:
    """Data GMM + isotropic coverage at its centers + heavy-tailed defense.

    The isotropic component prevents thin data-covariance directions from
    undersampling the model's isotropic Gaussian tails. All densities are
    evaluated as the complete mixture, including the Student-t component.
    """
    def __init__(self, X, n_components, seed):
        self.seed = int(seed)
        self.gmm = GaussianMixture(n_components=n_components, covariance_type='full',
                                   random_state=self.seed).fit(X)
        if not self.gmm.converged_:
            raise RuntimeError('Importance-proposal GMM did not converge.')
        self.loc = X.mean(axis=0)
        self.d = X.shape[1]
        self.variance = max(float(np.var(X,axis=0,ddof=1).mean()), 1e-8)
        self.shape = 4.*self.variance*np.eye(self.d)
        self.df = 5.
        J = len(self.gmm.weights_)
        self.weights = np.r_[.2*self.gmm.weights_, np.full(J,.7/J), .1]
        self.name = '0.2 data GMM + 0.7 isotropic center mixture + 0.1 Student-t(df=5); iid Monte Carlo'
    def sample(self, n):
        if int(n) != n or n < 2:
            raise ValueError('At least two integer integration samples are required.')
        n = int(n)
        U = np.random.default_rng(self.seed).random((n, self.d+2))
        U = np.clip(U, np.finfo(float).eps, 1-np.finfo(float).eps)
        labels = np.minimum(np.searchsorted(np.cumsum(self.weights), U[:,0], side='right'), len(self.weights)-1)
        G = norm.ppf(U[:,1:self.d+1]);Z = np.empty((n,self.d));J=len(self.gmm.weights_)
        for j in range(J):
            use = labels == j
            Z[use] = self.gmm.means_[j] + G[use] @ np.linalg.cholesky(self.gmm.covariances_[j]).T
            use = labels == J+j
            Z[use] = self.gmm.means_[j] + G[use]*np.sqrt(self.variance)
        use = labels == 2*J
        radial = np.sqrt(self.df / chi2.ppf(U[use,-1], self.df))
        Z[use] = self.loc + G[use]*(2*np.sqrt(self.variance))*radial[:,None]
        return Z, labels
    def score_samples(self, Z):
        d2=((Z[:,None,:]-self.gmm.means_[None,:,:])**2).sum(axis=2)
        log_iso=logsumexp(-.5*d2/self.variance,axis=1)-np.log(len(self.gmm.weights_))-.5*self.d*np.log(2*np.pi*self.variance)
        log_t=multivariate_t.logpdf(Z,loc=self.loc,shape=self.shape,df=self.df)
        return logsumexp(np.vstack([np.log(.2)+self.gmm.score_samples(Z),
                                   np.log(.7)+log_iso,np.log(.1)+log_t]),axis=0)

def build_proposal(X, num_samples=20000, n_components=None, seed=42):
    X = np.asarray(X,dtype=float)
    if X.ndim != 2 or X.shape[0]<2 or X.shape[1]<1 or not np.isfinite(X).all():
        raise ValueError('Proposal requires finite data with at least two rows and one feature.')
    components = min(3,len(X)) if n_components is None else n_components
    proposal = DefensiveProposal(X,components,seed)
    Z,_ = proposal.sample(num_samples)
    return Z,proposal.score_samples(Z),proposal

def _unpack(theta, k, d):
    centers = theta[1:1+k*d].reshape(k, d)
    logits = np.r_[theta[1+k*d:], 0.0]
    return theta[0], centers, logits-logsumexp(logits)


def _pack(sigma, centers, w):
    w = np.asarray(w, dtype=float)
    if not np.isfinite(sigma) or sigma <= 0 or np.any(w <= 0) or not np.isfinite(w).all():
        raise ValueError('The density requires positive scale and strictly positive weights.')
    w = w/w.sum()
    return np.r_[np.log(sigma), np.asarray(centers).ravel(), np.log(w[:-1])-np.log(w[-1])]


def _terms(X, centers, logw, m):
    # Use log space; only distances at floating-point zero receive a floor.
    d2 = np.maximum(cdist(X, centers, 'sqeuclidean'), 1e-200)
    scores = -(np.log(d2)+logw)/(m-1)
    lse = logsumexp(scores, axis=1)
    return np.exp(scores-lse[:, None]), np.exp(-(m-1)*lse), d2


def memberships(X, centers, w, m):
    if m <= 1:
        raise ValueError('m must exceed 1.')
    return _terms(_data(X), np.asarray(centers), np.log(w), m)[0]


def nll_gradient(theta, X, Z, logq, m, k, compute_center_gradient=True):
    """IS NLL and analytic gradient in log-scale / center / weight-logit coordinates."""
    logsigma, centers, logw = _unpack(theta, k, X.shape[1])
    w = np.exp(logw)
    with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
        ux, hx, dx = _terms(X, centers, logw, m)
        uz, hz, dz = _terms(Z, centers, logw, m)
        ex, ez = hx*np.exp(-2*logsigma), hz*np.exp(-2*logsigma)
        lr = -ez-logq
        lse = logsumexp(lr)
        prob = np.exp(lr-lse)
        value = len(X)*(lse-np.log(len(Z))) + ex.sum()

        def gradient(A, U, e, d2, a):
            t = a[:, None]*e[:, None]*U/d2
            # Accumulate displacements directly: subtracting two large matrix
            # products loses precision when a center nearly equals a data point.
            if compute_center_gradient:
                displacement = centers[None, :, :]-A[:, None, :]
                gv = 2*np.einsum('ij,ijd->jd', t, displacement)
            else:
                gv = np.zeros_like(centers)
            gw = ((a*e)[:, None]*(U-w)).sum(axis=0)[:-1]
            return np.r_[-2*np.dot(a, e), gv.ravel(), gw]

        grad = gradient(X, ux, ex, dx, np.ones(len(X)))-gradient(Z, uz, ez, dz, len(X)*prob)
    if not np.isfinite(value) or not np.isfinite(grad).all():
        return np.inf, np.zeros_like(theta)
    return float(value), grad


def importance_diagnostics(fit, Z, logq):
    _, h, _ = _terms(Z, fit['centers'], np.log(fit['w']), fit['m'])
    lr = -h/fit['sigma']**2-logq
    lse = logsumexp(lr)
    p = np.exp(lr-lse)
    return dict(logZ=float(lse-np.log(len(Z))), ESS=float(1/np.sum(p*p)),
                ESS_fraction=float(1/(len(Z)*np.sum(p*p))), largest_IS_weight=float(p.max()))


def fit_fixed(X, Z, logq, m, k, initial_sigma=1., initial_centers=None,
              initial_w=None, seed=321, max_outer=200, epsilon=1e-5,
              delta=1e-5, inner_maxiter=1000, K_post=1000,
              sigma_min=0., weight_floor=0.):
    """Algorithm emfuzzy for one m: block updates, then joint refinement.

    max_outer is an explicit operational guard; hitting it is reported as
    nonconvergence, never silently treated as satisfying the stopping rule.
    For bounded k=2 fits, w1 is optimized directly on [floor,1-floor].
    Otherwise offset-softmax enforces the weight floor. L-BFGS-B constrains
    log(sigma) >= log(sigma_min) when a positive scale floor is declared.
    Zero floors recover the original parameter domain.
    """
    X, Z = _data(X), _data(Z)
    logq = np.asarray(logq, dtype=float)
    if (m <= 1 or not np.isfinite(m) or not 1 <= k <= len(X)
            or X.shape[1] != Z.shape[1] or logq.shape != (len(Z),)
            or not np.isfinite(logq).all() or max_outer < 1 or inner_maxiter < 1
            or K_post < 0 or epsilon < 0 or delta < 0):
        raise ValueError('Invalid model, proposal, or optimizer configuration.')
    d = X.shape[1]
    centers = (X[np.random.RandomState(seed).choice(len(X), k, replace=False)].copy()
               if initial_centers is None else np.array(initial_centers, dtype=float, copy=True))
    if centers.shape != (k, d):
        raise ValueError('Initial centers must have shape (k,d).')
    w = np.full(k, 1/k) if initial_w is None else np.asarray(initial_w, dtype=float)
    if w.shape != (k,):
        raise ValueError('Initial weights must have shape (k,).')
    if not np.isfinite([sigma_min,weight_floor]).all() or sigma_min<0 or weight_floor<0 or k*weight_floor>=1:
        raise ValueError('Require sigma_min >= 0 and 0 <= weight_floor < 1/k.')
    if initial_sigma<sigma_min:
        raise ValueError('Initial parameters must satisfy the declared bounds.')
    if not np.isfinite(w).all() or np.any(w<=0):
        raise ValueError('Initial weights must be finite and positive.')
    w=w/w.sum()
    if np.any(w<weight_floor-1e-12):raise ValueError('Normalized initial weights violate weight_floor.')
    span=1-k*weight_floor
    residual=np.maximum((w-weight_floor)/span,np.finfo(float).eps)
    residual/=residual.sum()
    theta=_pack(initial_sigma,centers,residual)
    binary_direct=bool(k==2 and weight_floor>0)
    if binary_direct:theta[-1]=w[0]
    def model_coordinates(t):
        if binary_direct:
            # Direct bounded simplex coordinate avoids vanishing offset-logit
            # gradients when an active weight bound should be released.
            weights=np.array([t[-1],1-t[-1]])
            return np.r_[t[0],t[1:1+k*d],np.log(weights[0])-np.log(weights[1])]
        ls,v,lp=_unpack(t,k,d)
        # Stay in log-scale coordinates during line-search trials. Expanding
        # sigma and taking its logarithm again can overflow at valid trial ls.
        logweights=np.logaddexp(np.log(weight_floor),np.log(span)+lp) if weight_floor else lp
        return np.r_[ls,v.ravel(),logweights[:-1]-logweights[-1]]
    def objective(t,X,Z,logq,m,k,compute_center_gradient=True):
        actual=model_coordinates(t)
        value,g=nll_gradient(actual,X,Z,logq,m,k,compute_center_gradient)
        if binary_direct:
            g[-1]/=t[-1]*(1-t[-1])
        elif weight_floor:
            _,_,lp=_unpack(t,k,d);prob=np.exp(lp);weights=weight_floor+span*prob
            full=np.r_[g[1+k*d:],-g[1+k*d:].sum()]/weights
            g[1+k*d:]=(span*prob*(full-np.dot(prob,full)))[:-1]
        return value,g
    value,_=objective(theta,X,Z,logq,m,k)
    if not np.isfinite(value):raise RuntimeError('Initial likelihood is not finite.')
    sw_indices=np.r_[0,np.arange(1+k*d,len(theta))]
    bounds=[(np.log(sigma_min),None) if sigma_min>0 else (None,None)]+[(None,None)]*(len(theta)-1)
    if binary_direct:bounds[-1]=(weight_floor,1-weight_floor)
    history, converged, failures = [], False, 0

    def physical(t):
        ls, v, lw = _unpack(model_coordinates(t), k, d)
        return np.r_[np.exp(ls), v.ravel(), np.exp(lw)]

    for iteration in range(1, max_outer+1):
        old_theta, old_value = theta.copy(), value
        _, v, lw = _unpack(model_coordinates(theta), k, d)
        U, _, _ = _terms(X, v, lw, m)

        logum = m*np.log(np.maximum(U, np.finfo(float).tiny))
        um = np.exp(logum-logsumexp(logum, axis=0))
        theta[1:1+k*d] = (um.T@X).ravel()

        def block_objective(sw):
            candidate = theta.copy()
            candidate[sw_indices] = sw
            f, g = objective(candidate, X, Z, logq, m, k, compute_center_gradient=False)
            return f, g[sw_indices]

        opt = minimize(block_objective, theta[sw_indices], jac=True, method='L-BFGS-B',
                       bounds=[bounds[i] for i in sw_indices], options=dict(maxiter=inner_maxiter, ftol=1e-12, gtol=1e-6, maxls=100))
        if not opt.success:
            failures += 1
        if not np.isfinite(opt.fun):
            raise RuntimeError('Nonfinite scale/weight optimization.')
        theta[sw_indices] = opt.x
        value, _ = objective(theta, X, Z, logq, m, k)
        step = float(np.linalg.norm(physical(theta)-physical(old_theta)))
        change = abs(value-old_value)
        history.append(dict(stage='blockwise', iteration=iteration, NLL=value,
                            parameter_change=step, NLL_change=change,
                            optimizer_success=bool(opt.success), message=str(opt.message)))
        converged = bool(opt.success and (step <= epsilon or change <= delta))
        if converged:
            break

    post_success, post_message, post_iterations, post_status = True, 'K_post=0: refinement skipped', 0, 0
    post_attempts = 0
    # A fresh L-BFGS history can recover from an abnormal line search.
    if K_post:
        post_success = False
        for attempt, line_search_budget in enumerate((100, 200), start=1):
            remaining = K_post-post_iterations
            if remaining <= 0:
                break
            opt = minimize(objective, theta, args=(X, Z, logq, m, k), jac=True,
                           method='L-BFGS-B', bounds=bounds, options=dict(maxiter=remaining,
                           ftol=1e-12, gtol=1e-6, maxls=line_search_budget))
            trial_value, trial_gradient = objective(opt.x, X, Z, logq, m, k)
            accepted = bool(np.isfinite(trial_value) and trial_value <= value+1e-7)
            if accepted:
                theta, value = opt.x.copy(), float(trial_value)
            post_iterations += int(opt.nit)
            post_attempts = attempt
            post_success = bool(opt.success and accepted)
            post_status = int(opt.status) if accepted else 2
            post_message = str(opt.message) if accepted else 'Refinement returned a nonfinite or worsening point.'
            history.append(dict(stage='joint_refinement', iteration=post_iterations,
                                attempt=attempt, attempt_iterations=int(opt.nit),
                                NLL=value, optimizer_success=post_success,
                                optimizer_status=post_status, message=post_message,
                                gradient_inf_norm=float(np.max(np.abs(trial_gradient))),
                                accepted=accepted, maxls=line_search_budget))
            if post_success or post_status == 1:
                break
    ls, centers, lw = _unpack(model_coordinates(theta), k, d)
    result = dict(sigma=float(np.exp(ls)), centers=centers.copy(), w=np.exp(lw),
                  U=_terms(X, centers, lw, m)[0], m=float(m), k=int(k), NLL=float(value),
                  theta=model_coordinates(theta), history=history, converged=converged,
                  optimizer_failures=failures, post_success=post_success,
                  post_message=post_message, post_iterations=post_iterations,
                  post_status=post_status, post_attempts=post_attempts,
                  seed=int(seed), n_iter=iteration, engine='manuscript',
                  fit_options=dict(max_outer=max_outer, epsilon=epsilon, delta=delta,
                                   inner_maxiter=inner_maxiter, K_post=K_post,
                                   sigma_min=sigma_min,weight_floor=weight_floor))
    result['IS_diagnostics'] = importance_diagnostics(result, Z, logq)
    return result


def select_m(X, k, m_grid, num_is=20000, proposal_components=None,
             proposal_seed=674, seeds=(321,), **fit_options):
    """Fixed k; fit each m, refine, and select the smallest NLL (no XB filter)."""
    X = _data(X)
    grid = np.asarray(m_grid, dtype=float)
    if grid.ndim != 1 or not len(grid) or not np.isfinite(grid).all() or np.any(grid <= 1):
        raise ValueError('m_grid must contain finite values greater than one.')
    if not len(seeds):
        raise ValueError('Provide at least one initialization seed.')
    Z, logq, proposal = build_proposal(X, num_is, proposal_components, proposal_seed)
    fits, rows = [], []
    for m in grid:
        candidates = [fit_fixed(X, Z, logq, float(m), k, seed=int(s), **fit_options) for s in seeds]
        fit = min(candidates, key=lambda f: f['NLL'])
        fits.append(fit)
        rows.append(dict(m=float(m), NLL=fit['NLL'], converged=fit['converged'],
                         optimizer_failures=fit['optimizer_failures'],
                         post_success=fit['post_success'], post_status=fit['post_status'],
                         post_message=fit['post_message'], post_iterations=fit['post_iterations'],
                         post_attempts=fit['post_attempts'], seed=fit['seed']))
    best = min(fits, key=lambda f: f['NLL'])
    return best, rows, dict(samples=Z, logpdf=logq, model=proposal), fits


def sample_iid(fit, n, seed=123, max_proposals=10_000_000):
    """Independent rejection draws, with a global analytic Gaussian envelope.

    Let a_j = w_j ||x-v_j||^2 and C=k**(m-1). Then
    h(x) >= min_j(a_j)/C, hence exp(-h/sigma^2) <=
    sum_j exp(-w_j ||x-v_j||^2/(sigma^2*C)). Each summand is an
    unnormalized isotropic Gaussian. Select its component proportional to
    its integral and accept with exp(-h/sigma^2)/sum_j exp(-a_j/(sigma^2*C)).
    No fitted normalizing constant, burn-in, or MCMC approximation is needed.
    """
    if n < 1 or int(n) != n:
        raise ValueError('n must be a positive integer.')
    n = int(n)
    rng = np.random.default_rng(seed)
    centers, w = np.asarray(fit['centers']), np.asarray(fit['w'])
    sigma, m = fit['sigma'], fit['m']
    _pack(sigma, centers, w)
    if m <= 1 or not np.isclose(w.sum(), 1):
        raise ValueError('Invalid fitted model.')
    k, d = centers.shape
    logvar = 2*np.log(sigma)+(m-1)*np.log(k)-np.log(w)-np.log(2.)
    logmass = .5*d*logvar
    component_prob = np.exp(logmass-logsumexp(logmass))
    sd = np.exp(.5*logvar)
    chunks, accepted, proposed = [], 0, 0
    while accepted < n:
        count = min(max(256, 2*(n-accepted)), 8192, max_proposals-proposed)
        if count <= 0:
            raise RuntimeError('Rejection sampler proposal budget exceeded; no shortened dataset is returned.')
        labels = rng.choice(k, size=count, p=component_prob)
        Y = centers[labels]+rng.normal(size=(count, d))*sd[labels, None]
        _, h, _ = _terms(Y, centers, np.log(w), m)
        d2 = cdist(Y, centers, 'sqeuclidean')
        log_envelope = logsumexp(-d2/(2*np.exp(logvar)), axis=1)
        log_accept = -h/sigma**2-log_envelope
        if np.max(log_accept) > 1e-8:
            raise RuntimeError('Numerical violation of rejection envelope.')
        keep = Y[np.log(rng.random(count)) <= np.minimum(log_accept, 0)]
        chunks.append(keep)
        accepted += len(keep)
        proposed += count
    return np.concatenate(chunks)[:n]


def align_parameters(centers, w, observed_centers):
    rows, cols = linear_sum_assignment(cdist(observed_centers, centers, 'sqeuclidean'))
    order = np.empty(len(rows), dtype=int)
    order[rows] = cols
    return np.asarray(centers)[order].copy(), np.asarray(w)[order].copy(), order


def bootstrap_summary(estimates, k, d, alpha=.05, sigma_min=0., weight_floor=0.):
    A = np.asarray(estimates, dtype=float)
    if (A.ndim != 2 or A.shape[1] != 1+(d+1)*k or len(A) < max(d+1, k)
            or not np.isfinite(A).all() or not 0 < alpha < 1 or k < 2):
        raise ValueError('Invalid bootstrap array, B, alpha, or k (requires k >= 2).')
    if sigma_min<0 or weight_floor<0 or k*weight_floor>=1:
        raise ValueError('Invalid inference parameter bounds.')
    B = len(A)
    mean, sd = A.mean(axis=0), A.std(axis=0, ddof=0)
    z = norm.ppf(1-alpha/2)
    ci = np.column_stack([mean-z*sd, mean+z*sd])
    ci[0, 0] = max(sigma_min, ci[0, 0])  # endpoint 0 denotes intersection with sigma > 0
    ci[1+k*d:, :] = np.clip(ci[1+k*d:, :], weight_floor, 1-(k-1)*weight_floor)
    center_draws = A[:, 1:1+k*d].reshape(B, k, d)
    covariance = np.stack([np.atleast_2d(np.cov(center_draws[:, j, :], rowvar=False, ddof=1)) for j in range(k)])
    # The manuscript requires an inverse, not a center-covariance pseudoinverse.
    if any(np.linalg.matrix_rank(C) < d for C in covariance):
        raise RuntimeError('Singular center covariance: increase B/check failed or degenerate refits.')
    weight_cov = np.atleast_2d(np.cov(A[:, 1+k*d:], rowvar=False, ddof=1))
    if np.linalg.matrix_rank(weight_cov) < k-1:
        raise RuntimeError('Weight covariance has rank below k-1; check active bounds/degenerate refits. Increasing B alone may not resolve a structural rank deficiency.')
    return dict(B=B, k=k, d=d, alpha=alpha, mean=mean, sd=sd, ci=ci,
                scale_lower_open=bool(sigma_min==0 and ci[0, 0]==0),
                sigma_min=float(sigma_min),weight_floor=float(weight_floor),
                center_means=mean[1:1+k*d].reshape(k, d),
                center_covariance=covariance, center_precision=np.linalg.inv(covariance),
                center_cutoff=float(chi2.ppf(1-alpha, d)),
                weight_mean=mean[1+k*d:], weight_covariance=weight_cov,
                weight_precision=np.linalg.pinv(weight_cov, hermitian=True),
                weight_cutoff=float(chi2.ppf(1-alpha, k-1)))


def in_center_region(points, summary, center):
    delta = np.asarray(points)-summary['center_means'][center]
    return np.einsum('...i,ij,...j->...', delta, summary['center_precision'][center], delta) <= summary['center_cutoff']


def in_weight_region(weights, summary, atol=1e-10):
    W = np.asarray(weights, dtype=float)
    delta = W-summary['weight_mean']
    quadratic = np.einsum('...i,ij,...j->...', delta, summary['weight_precision'], delta)
    floor=summary.get('weight_floor',0.)
    upper=1-(summary['k']-1)*floor
    simplex = (W >= floor-atol).all(axis=-1) & (W <= upper+atol).all(axis=-1) & np.isclose(W.sum(axis=-1), 1., atol=atol, rtol=0.)
    return simplex & (quadratic <= summary['weight_cutoff'])


def parametric_bootstrap(X, fit, B=100, alpha=.05, num_is=20000,
                         proposal_components=None, seed=999, fit_options=None):
    """Algorithm inference: iid data, rebuilt proposal/cloud, fixed k,m, alignment."""
    X = _data(X)
    k, d = fit['centers'].shape
    if k < 2 or B < max(d+1, k) or int(B) != B or not 0 < alpha < 1:
        raise ValueError('Require k >= 2, integer B >= max(d+1,k), and 0 < alpha < 1.')
    if (X.shape[1] != d or not fit['converged'] or fit['optimizer_failures']
            or (not fit['post_success'] and fit['post_status'] != 1)):
        raise RuntimeError('Check dimensions and observed blockwise convergence before bootstrapping.')
    options = dict(fit.get('fit_options', {}))
    if fit_options:
        options.update(fit_options)
    for key in ('sigma_min','weight_floor'):
        if options.get(key,0.) != fit.get('fit_options',{}).get(key,0.):
            raise ValueError('Bootstrap parameter bounds must match the observed fitting domain.')
    rng = np.random.default_rng(seed)
    estimates, diagnostics = [], []
    for b in range(int(B)):
        sample_seed, proposal_seed, refit_seed = map(int, rng.integers(0, 2**31-1, size=3))
        Y = sample_iid(fit, len(X), seed=sample_seed)
        Z, logq, _ = build_proposal(Y, num_is, proposal_components, proposal_seed)
        refit = fit_fixed(Y, Z, logq, fit['m'], k, initial_sigma=fit['sigma'],
                          initial_centers=fit['centers'], initial_w=fit['w'],
                          seed=refit_seed, **options)
        if (not refit['converged'] or refit['optimizer_failures']
                or (not refit['post_success'] and refit['post_status'] != 1)):
            raise RuntimeError(f'Bootstrap replicate {b+1} did not converge cleanly; no partial inference returned.')
        centers, w, order = align_parameters(refit['centers'], refit['w'], fit['centers'])
        estimates.append(np.r_[refit['sigma'], centers.ravel(), w])
        diagnostics.append(dict(replicate=b+1, sample_seed=sample_seed,
                                proposal_seed=proposal_seed, refit_seed=refit_seed,
                                permutation=order.tolist(), NLL=refit['NLL'],
                                post_success=refit['post_success'],
                                IS_diagnostics=refit['IS_diagnostics']))
        print(f'Bootstrap {b+1}/{B}: fixed k={k}, m={fit["m"]:.6g}', flush=True)
    estimates = np.asarray(estimates)
    return dict(estimates=estimates, summary=bootstrap_summary(estimates, k, d, alpha,
                sigma_min=options.get('sigma_min',0.),weight_floor=options.get('weight_floor',0.)), diagnostics=diagnostics)

from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.special import logsumexp



def fit_is_clean(fit):
    return bool(fit['converged'] and not fit['optimizer_failures'] and
                (fit['post_success'] or fit['post_status']==1))




def final_partition_diagnostics(fit, min_hard_count=5):
    """Application quality diagnostic; never changes the NLL-selected m or fitted U."""
    counts=np.bincount(np.asarray(fit['U']).argmax(axis=1),minlength=len(fit['w']))
    return dict(hard_counts=counts.tolist(),min_hard_count=int(counts.min()),
                required_min_hard_count=int(min_hard_count),
                passes=bool(np.all(counts>=min_hard_count)))



def report_inference(bootstrap, output_dir):
    summary = bootstrap['summary']
    k, d = summary['k'], summary['d']
    names = ['sigma']+[f'v{a+1}_{j+1}' for a in range(k) for j in range(d)]+[f'w{a+1}' for a in range(k)]
    table = pd.DataFrame({'parameter': names, 'bootstrap_mean': summary['mean'],
                          'bootstrap_sd_ddof0': summary['sd'],
                          'lower': summary['ci'][:, 0], 'upper': summary['ci'][:, 1]})
    display(table)
    if summary['scale_lower_open']:
        print('The scale interval lower endpoint 0 is excluded (sigma > 0).')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir/'bootstrap_scalar_intervals.csv', index=False)
    np.savez_compressed(output_dir/'bootstrap_regions.npz', estimates=bootstrap['estimates'],
                        **{key: value for key, value in summary.items()})
    (output_dir/'bootstrap_diagnostics.json').write_text(json.dumps(bootstrap['diagnostics'], indent=2))
    print('Regions saved to', output_dir/'bootstrap_regions.npz')
    print('Use in_center_region(points, summary, center_index) and in_weight_region(weights, summary).')
    return summary, table

def parametric_bootstrap_parallel(X,fit,B=100,alpha=.05,num_is=100000,
                                  proposal_components=2,seed=999,workers=4,
                                  checkpoint_dir=None,fit_options=None):
    from concurrent.futures import ThreadPoolExecutor,as_completed
    if int(workers)!=workers or workers<1:
        raise ValueError('workers must be a positive integer.')
    if workers==1:
        return parametric_bootstrap(X,fit,B,alpha,num_is,proposal_components,seed,fit_options=fit_options)
    X=_data(X);k,d=fit['centers'].shape
    if B<max(d+1,k) or int(B)!=B or not 0<alpha<1 or not fit_is_clean(fit):
        raise ValueError('Require valid B/alpha and a clean observed fit.')
    options=dict(fit['fit_options'])
    if fit_options:options.update(fit_options)
    for key in ('sigma_min','weight_floor'):
        if options.get(key,0.)!=fit['fit_options'].get(key,0.):
            raise ValueError('Bootstrap parameter bounds must match the observed fitting domain.')
    rng=np.random.default_rng(seed)
    seed_triples=rng.integers(0,2**31-1,size=(int(B),3))
    estimates=[None]*int(B);diagnostics=[None]*int(B)
    checkpoint=Path(checkpoint_dir) if checkpoint_dir is not None else None
    if checkpoint is not None:checkpoint.mkdir(parents=True,exist_ok=True)
    def one_replicate(b):
        sample_seed,proposal_seed,refit_seed=map(int,seed_triples[b])
        Y=sample_iid(fit,len(X),seed=sample_seed)
        Z,logq,_=build_proposal(Y,num_is,proposal_components,proposal_seed)
        refit=fit_fixed(Y,Z,logq,fit['m'],k,initial_sigma=fit['sigma'],
            initial_centers=fit['centers'],initial_w=fit['w'],seed=refit_seed,**options)
        if not fit_is_clean(refit):
            detail={key:refit[key] for key in ('converged','optimizer_failures','n_iter','post_status','post_message','history')}
            if checkpoint is not None:
                (checkpoint/f'failed_replicate_{b+1:03d}.json').write_text(json.dumps(detail,indent=2))
            raise RuntimeError(f'Bootstrap replicate {b+1} did not converge cleanly: '
                f'outer={refit["n_iter"]}, converged={refit["converged"]}, '
                f'block_failures={refit["optimizer_failures"]}, post_status={refit["post_status"]}. '
                'No partial inference returned.')
        centers,w,order=align_parameters(refit['centers'],refit['w'],fit['centers'])
        estimate=np.r_[refit['sigma'],centers.ravel(),w]
        diagnostic=dict(replicate=b+1,sample_seed=sample_seed,proposal_seed=proposal_seed,
            refit_seed=refit_seed,permutation=order.tolist(),NLL=refit['NLL'],
            post_success=refit['post_success'],IS_diagnostics=refit['IS_diagnostics'])
        return b,estimate,diagnostic
    # Checkpoints are output only; this function never loads previous results.
    with ThreadPoolExecutor(max_workers=int(workers)) as pool:
        pending=[pool.submit(one_replicate,b) for b in range(int(B))]
        try:
            for completed,future in enumerate(as_completed(pending),1):
                b,estimate,diagnostic=future.result()
                estimates[b]=estimate;diagnostics[b]=diagnostic
                if checkpoint is not None:
                    np.savez_compressed(checkpoint/f'replicate_{b+1:03d}.npz',estimate=estimate)
                    (checkpoint/f'replicate_{b+1:03d}.json').write_text(json.dumps(diagnostic,indent=2))
                print(f'Bootstrap {completed}/{B} completed (replicate {b+1}); fixed k={k}, m={fit["m"]:.6g}',flush=True)
        except BaseException:
            for future in pending:future.cancel()
            raise
    estimates=np.asarray(estimates)
    return dict(estimates=estimates,summary=bootstrap_summary(estimates,k,d,alpha,
        sigma_min=options.get('sigma_min',0.),weight_floor=options.get('weight_floor',0.)),
        diagnostics=diagnostics)


In [ ]:
array_scaled = df_scaled.to_numpy()


In [ ]:

K = 2
M_GRID = np.geomspace(1.1, 6, 8)
NUM_IS, K_POST = 500000, 1000
INITIALIZATION_SEED, PROPOSAL_SEED = 32, 674
OUTPUT_DIR = Path('results_adni_final').resolve()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
SIGMA_MIN, WEIGHT_FLOOR = 0.05, 0.01
FIT_OPTIONS = dict(max_outer=2000,epsilon=1e-5,delta=1e-5,inner_maxiter=1000,K_post=K_POST,
                   sigma_min=SIGMA_MIN,weight_floor=WEIGHT_FLOOR)
data = array_scaled
Z, logq, proposal = build_proposal(data,NUM_IS,n_components=2,seed=PROPOSAL_SEED)
importance = dict(samples=Z,logpdf=logq,model=proposal)

from concurrent.futures import ThreadPoolExecutor
OBSERVED_WORKERS=4
(OUTPUT_DIR/'observed_checkpoints').mkdir(exist_ok=True)
def fit_m_endpoint(m):
    f=fit_fixed(data,Z,logq,float(m),K,seed=INITIALIZATION_SEED,**FIT_OPTIONS)
    import pickle
    with (OUTPUT_DIR/'observed_checkpoints'/f'm_{float(m):.12g}.pkl').open('wb') as handle:
        pickle.dump(f,handle)  # output only; never loaded by this notebook
    print('Fixed-k m fit:',m,'NLL:',f['NLL'],flush=True)
    return f
with ThreadPoolExecutor(max_workers=OBSERVED_WORKERS) as pool:
    candidate_fits=list(pool.map(fit_m_endpoint,M_GRID))
search_table=pd.DataFrame([dict(m=f['m'],NLL=f['NLL'],converged=f['converged'],optimizer_failures=f['optimizer_failures'],post_status=f['post_status'],post_iterations=f['post_iterations']) for f in candidate_fits])
search_table.to_csv(OUTPUT_DIR/'m_selection_nll.csv',index=False);display(search_table)
best_result=min(candidate_fits,key=lambda f:f['NLL'])
best_c,best_m=K,best_result['m'];result_fit=result_c=best_result
samples_IS,proposal_pdf=Z,np.exp(logq)
param_names=['sigma']+[f'v{a+1}_{j+1}' for a in range(K) for j in range(data.shape[1])]+[f'w{a+1}' for a in range(K)]
is_diagnostics_table=pd.DataFrame([dict(m=f['m'],**f['IS_diagnostics']) for f in candidate_fits])
is_diagnostics_table.to_csv(OUTPUT_DIR/'importance_diagnostics.csv',index=False)
np.savez_compressed(OUTPUT_DIR/'observed_fit.npz',sigma=best_result['sigma'],centers=best_result['centers'],w=best_result['w'],m=best_m,U=best_result['U'])
final_diagnostics=final_partition_diagnostics(best_result,5)
run_settings=dict(preprocessing='PCA of 12 clinical predictors; first three PCs retained',pca_provenance=pca_provenance,importance_cloud_sha256=__import__('hashlib').sha256(Z.tobytes()+logq.tobytes()).hexdigest(),input_file_sha256=__import__('hashlib').sha256(DATA_PATH.read_bytes()).hexdigest(),analysis_pc_columns=PC_COLUMNS,analysis_dimension=D,parameter_bounds=dict(sigma_min=SIGMA_MIN,weight_floor=WEIGHT_FLOOR),proposal=proposal.name,integration_draws='conditionally iid',bootstrap_completed=False,final_partition=final_diagnostics,selection='minimum refined NLL over m at scientifically fixed k=2',k=K,fixed_k=K,k_source='scientific modeling choice after exploratory analyses',k_selection_performed=False,m_grid=M_GRID.tolist(),selected_m=best_m,NLL=best_result['NLL'],num_is=NUM_IS,observed_workers=OBSERVED_WORKERS,proposal_components=2,proposal_seed=PROPOSAL_SEED,initialization_seed=INITIALIZATION_SEED,initialization_rng='numpy.random.RandomState',fit_options=FIT_OPTIONS,data_path=str(DATA_PATH.resolve()),input_sha256=__import__('hashlib').sha256(data.tobytes()).hexdigest(),bootstrap_status='not run',conditioning='fixed k=2, selected m and standardized PCs')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
print('Fixed k:',K,'Selected m:',best_m)

partition_table=pd.DataFrame([dict(m=f['m'],sigma=f['sigma'],min_weight=float(f['w'].min()),**final_partition_diagnostics(f,5)) for f in candidate_fits])
partition_table.to_csv(OUTPUT_DIR/'m_partition_diagnostics.csv',index=False);display(partition_table)

if not all(fit_is_clean(f) for f in candidate_fits):
    run_settings.update(selected_m=None,candidate_m_min_NLL=best_m,bootstrap_status='blocked: unresolved m candidate fit')
    (OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
    raise RuntimeError('Unresolved m candidate fitting problem; inspect candidate diagnostics. No inference produced.')

# Independent iid integration checks at every fitted m endpoint.
AUDIT_NUM_IS = 2000000
Za,qa,_=build_proposal(data,AUDIT_NUM_IS,n_components=2,seed=675)
audit_rows=[]
for f in candidate_fits:
    logtotal=-np.inf;logtotal2=-np.inf
    for start in range(0,len(Za),25000):
        stop=min(start+25000,len(Za))
        _,h,_=_terms(Za[start:stop],f['centers'],np.log(f['w']),f['m'])
        lw=-h/f['sigma']**2-qa[start:stop]
        logtotal=np.logaddexp(logtotal,logsumexp(lw))
        logtotal2=np.logaddexp(logtotal2,logsumexp(2*lw))
    logZ=logtotal-np.log(len(Za))
    relative_var=max(0.,np.exp(logtotal2-np.log(len(Za))-2*logZ)-1.)
    se=len(data)*np.sqrt(relative_var/(len(Za)-1))
    _,hd,_=_terms(data,f['centers'],np.log(f['w']),f['m'])
    audit_nll=float(hd.sum()/f['sigma']**2+len(data)*logZ)
    audit_rows.append(dict(m=f['m'],fitting_NLL=f['NLL'],independent_NLL=audit_nll,estimated_NLL_MC_se=se,NLL_difference=audit_nll-f['NLL']))
audit_table=pd.DataFrame(audit_rows);audit_table.to_csv(OUTPUT_DIR/'independent_normalizer_check.csv',index=False);display(audit_table)
audit_m=float(audit_table.loc[audit_table.independent_NLL.idxmin(),'m'])
run_settings.update(audit_num_is=AUDIT_NUM_IS,audit_seed=675,independent_m=audit_m,endpoint_ranking_agrees=bool(audit_m==best_m))
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
fig,ax=plt.subplots(figsize=(8,5));ax.plot(search_table.m,search_table.NLL,'o-',label='Fitting-cloud NLL');ax.errorbar(audit_table.m,audit_table.independent_NLL,yerr=2*audit_table.estimated_NLL_MC_se,fmt='s--',label='Independent NLL ± 2 MC SE');ax.set(xlabel='Fuzzifier m',ylabel='Negative log likelihood',title=f'm selection at fixed k={K}');ax.legend();fig.tight_layout();fig.savefig(OUTPUT_DIR/'m_selection_nll.png',dpi=160);plt.show()
if audit_m!=best_m:
    run_settings['bootstrap_status']='blocked: independent endpoint m ranking disagrees'
    (OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
    raise RuntimeError('Independent integration changes the endpoint m ranking. Increase integration budget and reassess before inference.')



In [ ]:
fig=plot_pc_fit(array_scaled,best_result['centers'],K,best_m)
(OUTPUT_DIR/'figures').mkdir(exist_ok=True)
plt.gcf().savefig(OUTPUT_DIR/'figures/adni_fitted_centers.png',dpi=180,bbox_inches='tight')
plt.show()


In [ ]:
from matplotlib.lines import Line2D
# Clinical labels are post-fit annotations only. Missing diagnoses are not non-AD.
label_known=df['cluster'].isin(['AD','Dementia','CN','MCI','EMCI','LMCI','SMC']).to_numpy()
true_labels=np.where(label_known,df['cluster'].isin(['AD','Dementia']).astype(float),np.nan)
if np.unique(true_labels[label_known]).size!=2:raise ValueError('Both AD and non-AD labels are required for annotation.')
U=best_result['U'];ad_fraction_wfcm=U[label_known].T@true_labels[label_known]/U[label_known].sum(axis=0)
AD_index_wfcm=int(ad_fraction_wfcm.argmax());soft_weights_AD=U[:,AD_index_wfcm]
WFCM_COLOR_MIN,WFCM_COLOR_MAX=map(float,np.quantile(soft_weights_AD,[.05,.95]))
if WFCM_COLOR_MAX<=WFCM_COLOR_MIN:WFCM_COLOR_MIN,WFCM_COLOR_MAX=0.,1.
run_settings['display_contrast']=dict(lower=WFCM_COLOR_MIN,upper=WFCM_COLOR_MAX,rule='fitted-membership 5th and 95th percentiles; no label tuning',colorbar='raw membership units; clipped display')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
soft_weights_AD_display=np.clip((soft_weights_AD-WFCM_COLOR_MIN)/(WFCM_COLOR_MAX-WFCM_COLOR_MIN),0,1)
# Flip PC1 and PC3 for display.
plot_signs = np.array([-1., 1., -1.])
plot_scores = array_scaled * plot_signs
plot_centers = best_result['centers'] * plot_signs   
fig=compare_pc_memberships(array_scaled,[(soft_weights_AD,best_result['centers'],'WFCM Soft AD Spectrum',True,False),(true_labels,best_result['centers'],'Hard Binary Observed AD / non-AD labels',False,True)],WFCM_COLOR_MIN,WFCM_COLOR_MAX)
print('Raw membership quantiles:',np.quantile(soft_weights_AD,[0,.05,.5,.95,1]))
annotation_table=pd.DataFrame(dict(cluster=np.arange(K)+1,hard_count=np.bincount(U.argmax(1),minlength=K),membership_weighted_AD_fraction=ad_fraction_wfcm));annotation_table.to_csv(OUTPUT_DIR/'postfit_annotation_diagnostics.csv',index=False);display(annotation_table)
(OUTPUT_DIR/'figures').mkdir(exist_ok=True)
plt.gcf().savefig(OUTPUT_DIR/'figures/adni_membership_vs_labels.png',dpi=180,bbox_inches='tight')
plt.show()


In [ ]:
# Two-cluster line or generic membership heatmap, depending on selected K.
U=best_result['U']
if K==2:
    nonAD_index_wfcm=1-AD_index_wfcm
    corners=np.array([[0.,0.],[1.,0.]])
    points=U[:,[AD_index_wfcm,nonAD_index_wfcm]]@corners
    colors_rgb=np.zeros((len(U),3));colors_rgb[:,0]=U[:,AD_index_wfcm];colors_rgb[:,2]=U[:,nonAD_index_wfcm]


In [ ]:
if K==2:
    # Plot
    fig, ax = plt.subplots(figsize=(8, 10))
    
    # Draw line segment
    ax.plot(corners[:, 0], corners[:, 1], 'k-', alpha=0.3)
    
    # Plot cluster centers
    ax.scatter(corners[:, 0], corners[:, 1],
               c=['red', 'blue'], s=100, label='Annotated cluster vertices')
    
    # Plot data points with soft membership coloring
    ax.scatter(points[:, 0], points[:, 1],
               c=colors_rgb, s=50, alpha=0.8, edgecolor='k', linewidth=0.2)
    
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 0.1)
    ax.set_yticks([])  
    ax.set_xlabel('Membership to non-AD-like cluster (AD-like at 0)')
    ax.set_title('Soft Cluster Memberships on 2D Simplex (AD vs non-AD)', fontsize = 22, pad=20)
    
    # Legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='AD-like',
               markerfacecolor='red', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='non-AD-like',
               markerfacecolor='blue', markersize=10)
    ]
    ax.legend(handles=legend_elements, loc='upper right', title='Annotated cluster vertices')
    
    plt.tight_layout()
    None
else:
    fig,ax=plt.subplots(figsize=(8,5));im=ax.imshow(U[np.argsort(U.argmax(axis=1))].T,aspect='auto',vmin=0,vmax=1,cmap='viridis')
    ax.set(xlabel='Subjects sorted by fitted cluster',ylabel='Cluster',title='Actual fitted memberships');fig.colorbar(im,ax=ax,label='Membership');plt.tight_layout();None

(OUTPUT_DIR/'figures').mkdir(exist_ok=True)
plt.gcf().savefig(OUTPUT_DIR/'figures/adni_membership_simplex.png',dpi=180,bbox_inches='tight')
plt.show()


# FCM vs GMM

In [ ]:
from sklearn.metrics import adjusted_rand_score, accuracy_score, roc_auc_score

def xie_beni_index(data, membership, centroids, m):
    d2 = ((data[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    separations = ((centroids[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    np.fill_diagonal(separations, np.inf)
    minimum = separations.min()
    return float(np.sum(membership ** m * d2) / (len(data) * minimum)) if minimum > 0 else np.inf
best_gmm = GaussianMixture(n_components=K, covariance_type='full', random_state=12, n_init=5, max_iter=300).fit(array_scaled)
if not best_gmm.converged_:
    raise RuntimeError('The fixed-k GMM comparison failed to converge.')
best_bic = best_gmm.bic(array_scaled)
gmm_u = best_gmm.predict_proba(array_scaled)
wfcms_u = best_result['U']
wfcms_labels = wfcms_u.argmax(axis=1)
gmm_labels = gmm_u.argmax(axis=1)
ad_fraction_gmm = gmm_u[label_known].T @ true_labels[label_known] / gmm_u[label_known].sum(axis=0)
AD_index_gmm = int(np.argmax(ad_fraction_gmm))
wfcms_majority_pred = (ad_fraction_wfcm >= 0.5).astype(int)[wfcms_labels]
wfcms_pred = (wfcms_labels == AD_index_wfcm).astype(int)
gmm_majority_pred = (ad_fraction_gmm >= 0.5).astype(int)[gmm_labels]
gmm_pred = (gmm_labels == AD_index_gmm).astype(int)
xb_wfcm = xie_beni_index(array_scaled, wfcms_u, best_result['centers'], best_m)
xb_gmm = xie_beni_index(array_scaled, gmm_u, best_gmm.means_, best_m)
print(f'GMM components fixed at k={K}; descriptive BIC: {best_bic:.2f}')
print(f'Descriptive fuzzy XB (same exponent m={best_m:.6g}): WFCM={xb_wfcm:.4f}, GMM={xb_gmm:.4f}')

def percentile_threshold_ari(scores):
    lower, upper = np.quantile(scores, [0.05, 0.95])
    cutoff = (lower + upper) / 2
    predicted = (scores >= cutoff).astype(int)
    ari = adjusted_rand_score(observed, predicted[label_known])
    return ari, cutoff

wfcm_threshold_ari, wfcm_membership_cutoff = percentile_threshold_ari(
    wfcms_u[:, AD_index_wfcm]
)
gmm_threshold_ari, gmm_cutoff = percentile_threshold_ari(
    gmm_u[:, AD_index_gmm]
)

print(
    f'Percentile-midpoint ARI (in-sample): '
    f'WFCM={wfcm_threshold_ari:.3f} (cutoff={wfcm_membership_cutoff:.4f}), '
    f'GMM={gmm_threshold_ari:.3f} (cutoff={gmm_cutoff:.4f})'
)



print(f'In-sample one-to-one class-mapped accuracy: WFCM={accuracy_score(true_labels[label_known], wfcms_pred[label_known]):.3f}, GMM={accuracy_score(true_labels[label_known], gmm_pred[label_known]):.3f}')
print(f'Majority-label accuracy baseline: {max(np.nanmean(true_labels), 1 - np.nanmean(true_labels)):.3f}')
print('Descriptive label matching is not an independent validation or a k-selection criterion.')
descriptive_auc_wfcm = roc_auc_score(true_labels[label_known], wfcms_u[:, AD_index_wfcm][label_known])
descriptive_auc_gmm = roc_auc_score(true_labels[label_known], gmm_u[:, AD_index_gmm][label_known])
print(f'Descriptive in-sample membership-score AUC: WFCM={descriptive_auc_wfcm:.3f}, GMM={descriptive_auc_gmm:.3f}')
print('Post-fit label annotation and these in-sample scores do not establish external validity or longitudinal progression.')
pd.DataFrame([dict(n_labelled=int(label_known.sum()), n_unlabelled=int((~label_known).sum()), method='WFCM', fixed_k=K, membership_score_AUC=descriptive_auc_wfcm, ARI=adjusted_rand_score(true_labels[label_known], wfcms_labels[label_known]), label_mapped_accuracy=accuracy_score(true_labels[label_known], wfcms_pred[label_known]), accuracy_mapping='one-to-one AD-like versus non-AD-like', per_component_majority_accuracy=accuracy_score(true_labels[label_known], wfcms_majority_pred[label_known]), majority_baseline=max(np.nanmean(true_labels), 1 - np.nanmean(true_labels))), dict(n_labelled=int(label_known.sum()), n_unlabelled=int((~label_known).sum()), method='GMM', fixed_k=K, membership_score_AUC=descriptive_auc_gmm, ARI=adjusted_rand_score(true_labels[label_known], gmm_labels[label_known]), label_mapped_accuracy=accuracy_score(true_labels[label_known], gmm_pred[label_known]), accuracy_mapping='one-to-one AD-like versus non-AD-like', per_component_majority_accuracy=accuracy_score(true_labels[label_known], gmm_majority_pred[label_known]), majority_baseline=max(np.nanmean(true_labels), 1 - np.nanmean(true_labels)))]).to_csv(OUTPUT_DIR / 'descriptive_label_metrics.csv', index=False)


In [ ]:
fig=compare_pc_memberships(array_scaled,[(best_result['U'][:,AD_index_wfcm],best_result['centers'],'WFCM Soft AD Spectrum',True,False),(best_gmm.predict_proba(array_scaled)[:,AD_index_gmm],best_gmm.means_,'GMM Soft AD Spectrum',False,False)],WFCM_COLOR_MIN,WFCM_COLOR_MAX)
(OUTPUT_DIR/'figures').mkdir(exist_ok=True)
plt.gcf().savefig(OUTPUT_DIR/'figures/adni_wfcm_vs_gmm.png',dpi=180,bbox_inches='tight')
plt.show()


In [ ]:

if not final_diagnostics['passes']:
    print('Diagnostic: highest-membership counts',final_diagnostics['hard_counts'],
          'are imbalanced. This does not stop fixed-k parametric bootstrap or change m.')
B,ALPHA=100,0.05
BOOTSTRAP_MAX_OUTER=5000  # operational guard only; epsilon/delta and update equations unchanged
BOOTSTRAP_FIT_OPTIONS=dict(max_outer=BOOTSTRAP_MAX_OUTER)
BOOTSTRAP_NUM_IS=100000  # independent finite-cloud refit approximation; reported explicitly
BOOTSTRAP_WORKERS=4  # set 1 for serial execution; seed triples and replicate order are unchanged
run_settings.update(B=B,alpha=ALPHA,bootstrap_seed=999,bootstrap_num_is=BOOTSTRAP_NUM_IS,bootstrap_fit_options={**FIT_OPTIONS,**BOOTSTRAP_FIT_OPTIONS},bootstrap_status='running',
                    bootstrap_completed=False,hard_assignment_policy='diagnostic only for fixed-k fuzzy spectrum')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
try:
    bootstrap=parametric_bootstrap_parallel(data,best_result,B=B,alpha=ALPHA,num_is=BOOTSTRAP_NUM_IS,proposal_components=2,seed=999,workers=BOOTSTRAP_WORKERS,checkpoint_dir=OUTPUT_DIR/'bootstrap_checkpoints',fit_options=BOOTSTRAP_FIT_OPTIONS)
except Exception as error:
    run_settings.update(bootstrap_status='failed numerical or covariance check',bootstrap_failure=str(error))
    (OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
    raise
inference,interval_table=report_inference(bootstrap,OUTPUT_DIR)
boot_estimates=bootstrap['estimates'];param_means,param_stds=inference['mean'],inference['sd']
ci_lo,ci_hi=inference['ci'][:,0],inference['ci'][:,1]
cov_full=np.cov(boot_estimates,rowvar=False,ddof=1)
run_settings.update(bootstrap_status='complete',bootstrap_completed=True,bootstrap_workers=BOOTSTRAP_WORKERS,bootstrap_seed_policy='same per-replicate seed triples and ordering as serial')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))


In [ ]:
# Three-PC analysis: chi-square df=3 center confidence ellipsoids.
confidence_figure,paired_figure=save_adni_regions(array_scaled,best_result['U'],inference,AD_index_wfcm,OUTPUT_DIR/'figures')
table2=write_adni_table2(inference,OUTPUT_DIR);display(table2)
run_settings['confidence_region_dimension']=D
run_settings['confidence_figures']=dict(regions='figures/adni_ci.png',paired='figures/adni_centers_simplex.png' if D==2 else 'figures/adni_figure8_centers_simplex.png')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
plt.show()

parameter_comparison=table2.copy()
parameter_comparison.insert(1,'fitted_estimate',np.r_[best_result['sigma'],best_result['centers'].ravel(),best_result['w']])
parameter_comparison.to_csv(OUTPUT_DIR/'parameter_estimates_and_intervals.csv',index=False)
(OUTPUT_DIR/'model_parameters.json').write_text(json.dumps(dict(k=K,m=best_m,sigma=float(best_result['sigma']),centers_standardized_PC=best_result['centers'].tolist(),distance_weights=best_result['w'].tolist(),AD_like_component_index_1based=AD_index_wfcm+1,PC_columns=PC_COLUMNS,PC_standardization_mean=scaler.mean_.tolist(),PC_standardization_scale=scaler.scale_.tolist()),indent=2))


In [ ]:
from PIL import Image
from IPython.display import display

figures_dir = OUTPUT_DIR / "figures"

with Image.open(figures_dir / "adni_figure8_centers_simplex.png") as img:
    top = img.convert("RGBA")

with Image.open(figures_dir / "adni_ci_zoom.png") as img:
    bottom = img.convert("RGBA")

gap = 24  # Vertical spacing in pixels
width = max(top.width, bottom.width)
height = top.height + gap + bottom.height
combined = Image.new("RGB", (width, height), "white")

combined.paste(top, ((width - top.width) // 2, 0), top)
combined.paste(bottom, ((width - bottom.width) // 2, top.height + gap), bottom)

output_path = figures_dir / "adni_centers_simplex_zoom.png"
combined.save(output_path, dpi=(220, 220))
display(combined)